In [1]:
import os
import shutil
import glob
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

original_train_dir = '../Data/train_cropped/'
pseudo_train_dir = '../Data/train_pseudo/'

if not os.path.exists(pseudo_train_dir):
    print("Menduplikasi data training asli ke folder pseudo...")
    shutil.copytree(original_train_dir, pseudo_train_dir)
    print("Duplikasi selesai!")
else:
    print("Folder pseudo sudah ada, siap ditambahkan data test!")

Menggunakan device: cuda:0
Menduplikasi data training asli ke folder pseudo...
Duplikasi selesai!


### Seleksi Data Test

In [4]:
import pandas as pd

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
CONFIDENCE_THRESHOLD = 0.95 # Syarat: Model harus yakin 95% ke atas

transform_conv = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mtcnn = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)

# Load model ConvNeXt yang sudah pintar
model_conv = timm.create_model('convnext_tiny', pretrained=False, num_classes=len(class_names))
model_conv.load_state_dict(torch.load('../models/convnext_tiny_model.pth', map_location=device))
model_conv = model_conv.to(device)
model_conv.eval()

test_dir = '../Data/test/'
submission_df = pd.read_csv('../Data/samplesubmission.csv')

pseudo_added_count = 0

print(f"Memulai perburuan Pseudo-Label dengan threshold {CONFIDENCE_THRESHOLD * 100}%...")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files: continue
            
        img_path = matching_files[0] 
        img_ext = os.path.splitext(img_path)[1]
        
        try:
            img = Image.open(img_path).convert('RGB')
            face = mtcnn(img)
            
            # HANYA mengambil data yang wajahnya berhasil di-crop sempurna
            if face is not None:
                pil_img = transforms.ToPILImage()(face.byte())
                
                tensor_input = transform_conv(pil_img).unsqueeze(0).to(device)
                output = model_conv(tensor_input)
                
                # Hitung persentase keyakinan
                probs = F.softmax(output, dim=1)[0]
                max_prob, predicted_idx = torch.max(probs, 0)
                
                # Jika model SANGAT YAKIN, simpan gambar potongan wajah ke folder train_pseudo
                if max_prob.item() >= CONFIDENCE_THRESHOLD:
                    predicted_class = class_names[predicted_idx.item()]
                    
                    # Tentukan lokasi simpan
                    target_folder = os.path.join(pseudo_train_dir, predicted_class)
                    target_file = os.path.join(target_folder, f"pseudo_{img_id}{img_ext}")
                    
                    # Simpan gambar potongan wajah
                    pil_img.save(target_file)
                    pseudo_added_count += 1
                    
        except Exception as e:
            pass 

print("="*40)
print(f"Berhasil menambahkan {pseudo_added_count} gambar test ke dalam data training!")
print("="*40)

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

Memulai perburuan Pseudo-Label dengan threshold 95.0%...


  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Berhasil menambahkan 322 gambar test ke dalam data training!


### Melatih Ulang Model dengan Data Gabungan

In [5]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets

BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-4
IMG_SIZE = 224

# --- PAKAI DATASET PSEUDO ---
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=pseudo_train_dir, transform=val_transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_dataset.dataset.transform = train_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Inisialisasi ulang model dari nol (pre-trained ImageNet)
model_pseudo = timm.create_model('convnext_tiny', pretrained=True, num_classes=len(class_names), drop_rate=0.3)
model_pseudo = model_pseudo.to(device)

optimizer = optim.AdamW(model_pseudo.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

model_save_path = '../models/convnext_tiny_pseudo_model.pth'

best_val_acc = 0.0
patience = 3 
patience_counter = 0

print(f"Memulai Training ConvNeXt dengan Pseudo-Labels (Total Data: {len(full_dataset)})...")
for epoch in range(EPOCHS):
    model_pseudo.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model_pseudo(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * inputs.size(0)
        
    scheduler.step()
    epoch_train_loss = running_loss / len(train_dataset)
    
    model_pseudo.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_pseudo(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct / total
    
    print(f"Epoch {epoch+1} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model_pseudo.state_dict(), model_save_path)
        print(f"Model Pseudo terbaik disimpan! (Akurasi: {best_val_acc:.4f})")
        patience_counter = 0 
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print("Early Stopping aktif!")
        break

Memulai Training ConvNeXt dengan Pseudo-Labels (Total Data: 1763)...


Epoch 1/15 [Train]:   0%|          | 0/89 [00:00<?, ?it/s]

Epoch 1 | Val Loss: 0.7916 | Val Acc: 0.6827
Model Pseudo terbaik disimpan! (Akurasi: 0.6827)


Epoch 2/15 [Train]:   0%|          | 0/89 [00:00<?, ?it/s]

Epoch 2 | Val Loss: 0.3324 | Val Acc: 0.8839
Model Pseudo terbaik disimpan! (Akurasi: 0.8839)


...

Epoch 8 | Val Loss: 0.4133 | Val Acc: 0.8867


Epoch 9/15 [Train]:   0%|          | 0/89 [00:00<?, ?it/s]

Epoch 9 | Val Loss: 0.4060 | Val Acc: 0.9008


Epoch 10/15 [Train]:   0%|          | 0/89 [00:00<?, ?it/s]

Epoch 10 | Val Loss: 0.4400 | Val Acc: 0.8895
Early Stopping aktif!


In [7]:
import pandas as pd
import glob

# --- 1. LOAD MODEL HASIL PSEUDO-LABELING ---
model_pseudo = timm.create_model('convnext_tiny', pretrained=False, num_classes=len(class_names))
model_pseudo.load_state_dict(torch.load('../models/convnext_tiny_pseudo_model.pth', map_location=device))
model_pseudo = model_pseudo.to(device)
model_pseudo.eval()

# --- 2. PERSIAPAN MULTI-ZOOM MTCNN ---
mtcnn_tight = MTCNN(keep_all=False, select_largest=True, margin=20, post_process=False, device=device)
mtcnn_normal = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)
mtcnn_wide = MTCNN(keep_all=False, select_largest=True, margin=100, post_process=False, device=device)

test_dir = '../Data/test/' 
submission_df = pd.read_csv('../Data/samplesubmission.csv')
predictions = []

print("Memulai Prediksi Ultimate dengan Pseudo-ConvNeXt...")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files:
            predictions.append('fake_unknown')
            continue
            
        img_path = matching_files[0] 
        
        try:
            img = Image.open(img_path).convert('RGB')
            faces = [mtcnn_tight(img), mtcnn_normal(img), mtcnn_wide(img)]
            all_probs = []
            
            for i, face in enumerate(faces):
                if face is not None:
                    pil_img = transforms.ToPILImage()(face.byte())
                else:
                    w, h = img.size
                    ratio = 0.4 if i == 0 else (0.6 if i == 1 else 0.8)
                    margin = (1.0 - ratio) / 2.0
                    left, top = w * margin, h * margin
                    right, bottom = w * (1.0 - margin), h * (1.0 - margin)
                    pil_img = img.crop((left, top, right, bottom))
                    
                tensor_input = val_transform(pil_img).unsqueeze(0).to(device)
                output = model_pseudo(tensor_input)
                all_probs.append(F.softmax(output, dim=1))
            
            # Rata-ratakan tebakan dari 3 level zoom
            avg_prob = torch.mean(torch.stack(all_probs), dim=0)
            
            # Ambil kelas pemenang akhir
            _, predicted_idx = torch.max(avg_prob, 1)
            predicted_class = class_names[predicted_idx.item()]
            predictions.append(predicted_class)
            
        except Exception as e:
            print(f"Error memproses {img_path}: {e}")
            predictions.append('fake_unknown')

# --- 3. SIMPAN KE CSV ---
submission_dir = '../Data/submission/'
os.makedirs(submission_dir, exist_ok=True)
submission_df['label'] = predictions

file_name = 'submission_findit_pseudo_convnext.csv'
submission_file_path = os.path.join(submission_dir, file_name)
submission_df.to_csv(submission_file_path, index=False)

print("="*40)
print(f"File submission Pseudo-Label berhasil disimpan!")
print(f"Lokasi: {submission_file_path}")
print("="*40)
display(submission_df.head(10))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15584\2945854588.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_pseudo.load_state_dict(torch.load('../models/convnext

Memulai Prediksi Ultimate dengan Pseudo-ConvNeXt...


d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


File submission Pseudo-Label berhasil disimpan!
Lokasi: ../Data/submission/submission_findit_pseudo_convnext.csv


,id,label
0,test_001,fake_printed
1,test_002,fake_screen
2,test_003,fake_mask
3,test_004,realperson
4,test_005,realperson
5,test_006,fake_unknown
6,test_007,fake_mask
7,test_008,fake_mannequin
8,test_009,fake_mask
9,test_010,realperson
